# TP2 - Clasificación de Género Musical (Spotify)
### Tecnología Digital VI - Inteligencia Artificial

**Integrantes:** _(completar)_

Este notebook resuelve el problema de clasificación multiclase de género
musical de la competencia de Kaggle: predecir `track_genre` a partir de
métricas acústicas de Spotify (`danceability`, `energy`, `tempo`, etc.).

Estructura del notebook:

1. Carga de datos
2. Análisis Exploratorio de Datos (EDA)
3. Preprocesamiento
4. Modelos y búsqueda de hiperparámetros (optimizando F1-Macro)
5. Model Selection (comparación por cross-validation)
6. Métrica MAP@3 y comparación F1-Macro vs MAP@3
7. Optimización pensando en MAP@3
8. Discusión de negocio: qué métrica usar según el caso
9. Entrenamiento final y generación del CSV para Kaggle

La lógica reutilizable (preprocesamiento, métricas, modelos, tuning) vive
en `src/spotify_clf/` — este notebook la importa y se enfoca en el
análisis, los resultados y las decisiones tomadas.


In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/spotify_clf desde el notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from spotify_clf import config, data
from spotify_clf.preprocessing import build_preprocessor
from spotify_clf.metrics import (
    f1_macro_scorer,
    map_at_3_scorer,
    map_at_3_from_model,
    mean_average_precision_at_k,
    top_k_predictions,
)
from spotify_clf.models import get_model_registry
from spotify_clf.tuning import tune_model
from spotify_clf.evaluate import cross_validate_model, results_to_dataframe
from spotify_clf.submission import build_submission, save_submission

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = config.RANDOM_STATE


## 1. Carga de datos

Coloquen `train.csv` y `test.csv` de la competencia de Kaggle en
`data/raw/` (ver `README.md`). Si todavía no los tienen, pueden explorar
el esquema con `data/raw/dataset_reference.csv` (dataset original, **no**
es el de la competencia — no usar para entrenar el modelo final).

In [ ]:
# Si ya tenés train.csv / test.csv de Kaggle en data/raw/, usá estas líneas:
train_df = data.load_train()
test_df = data.load_test()

# Si todavía no los tenés y solo querés probar el pipeline con el dataset
# de referencia, descomentá esto (NO es la data real de la competencia):
# train_df = pd.read_csv(config.REFERENCE_PATH)

train_df.shape, train_df.head()


In [ ]:
train_df.info()


In [ ]:
train_df.isna().sum()


## 2. Análisis Exploratorio de Datos (EDA)

Objetivo: entender qué características diferencian mejor a los distintos
géneros, y detectar problemas de calidad de datos (duplicados, outliers,
escalas muy distintas entre features) que condicionen el preprocesamiento.

### 2.1 Distribución de clases (géneros)

In [ ]:
genre_counts = train_df[config.TARGET_COL].value_counts()
print(f"Cantidad de géneros: {genre_counts.shape[0]}")
print(genre_counts.describe())

fig, ax = plt.subplots(figsize=(12, 5))
genre_counts.head(30).plot(kind="bar", ax=ax)
ax.set_title("Cantidad de canciones por género (top 30)")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()


**A tener en cuenta:** si algunos géneros tienen muchas menos canciones
que otros, F1-Macro los penaliza igual que a los géneros mayoritarios
(promedia el F1 por clase sin ponderar por soporte) — vale la pena mirar
el balance de clases antes de elegir `class_weight` o técnicas de
resampling.

### 2.2 Duplicados

In [ ]:
n_exact_dupes = train_df.duplicated().sum()
n_id_dupes = train_df[config.ID_COL].duplicated().sum() if config.ID_COL in train_df else None
print(f"Filas exactamente duplicadas: {n_exact_dupes}")
print(f"track_id repetidos (misma canción en > 1 género): {n_id_dupes}")


En el dataset original de Spotify es común que la misma canción (mismo
`track_id`) aparezca etiquetada bajo más de un género (p. ej. una canción
"acoustic" que también está en "pop"). Esto es importante para el split
train/validation: si la misma canción cae en train y en validación con
etiquetas distintas, se está sobreestimando la performance real. Si
detectan esto en su `train.csv`, una opción es hacer el split agrupando
por `track_id` (`GroupShuffleSplit` / `StratifiedGroupKFold`) en vez de un
split aleatorio simple.

### 2.3 Distribución y correlación de features numéricas

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flat, config.NUMERIC_COLS):
    sns.histplot(train_df[col], ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
corr = train_df[config.NUMERIC_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre features numéricas")
plt.tight_layout()
plt.show()


### 2.4 ¿Qué features separan mejor a los géneros?

Comparamos la distribución de algunas features clave entre un subconjunto
de géneros con sonoridades bien distintas (para que el gráfico sea
legible con más de 100 clases posibles).

In [ ]:
sample_genres = [
    g for g in ["acoustic", "classical", "edm", "metal", "hip-hop", "jazz"]
    if g in train_df[config.TARGET_COL].unique()
]
if len(sample_genres) < 3:
    # fallback si el subset no está en el train.csv de la competencia
    sample_genres = list(train_df[config.TARGET_COL].value_counts().head(6).index)

subset = train_df[train_df[config.TARGET_COL].isin(sample_genres)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, ["danceability", "energy", "acousticness"]):
    sns.boxplot(data=subset, x=config.TARGET_COL, y=feat, ax=ax)
    ax.set_title(f"{feat} por género")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=subset, x="energy", y="acousticness", hue=config.TARGET_COL, alpha=0.4, s=15, ax=ax
)
ax.set_title("energy vs acousticness por género")
plt.tight_layout()
plt.show()


**Conclusiones del EDA** _(completar con lo que observen en su `train.csv`
real)_:

- ¿Qué features separan mejor a los géneros más "extremos" (ej. classical
  vs. metal)?
- ¿Hay géneros que se solapan mucho entre sí en el espacio de features
  (candidatos naturales a confundirse, y por eso buenos candidatos a
  aparecer en el top-3 de MAP@3 aunque no sean la predicción top-1)?
- ¿Alguna feature tiene una distribución muy sesgada que amerite
  transformarla (log, etc.)?

## 3. Preprocesamiento

Usamos el `ColumnTransformer` de `src/spotify_clf/preprocessing.py`:

- **Features numéricas continuas** (`popularity`, `duration_ms`,
  `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`,
  `instrumentalness`, `liveness`, `valence`, `tempo`) → `StandardScaler`.
  Necesario para modelos lineales (Logistic Regression) y basados en
  distancia (KNN); no perjudica a los modelos de árboles.
- **Features categóricas de baja cardinalidad** (`key`, `mode`,
  `time_signature`, `explicit`) → `OneHotEncoder`. Son códigos sin una
  relación de orden lineal real (ej. `key` es un pitch class 0-11), así
  que tratarlas como numéricas introduciría una relación de orden
  artificial.
- **Columnas de texto libre e identificadores** (`track_id`, `artists`,
  `album_name`, `track_name`) → se descartan del pipeline "out of the
  box". Quedan documentadas en `config.TEXT_COLS` para quien quiera
  experimentar con feature engineering de texto (cantidad de artistas
  colaboradores, longitud del nombre del track, etc.) como extensión.

El preprocesador se arma **dentro** de cada `Pipeline` de
`src/spotify_clf/models.py`, así que el `fit` del scaler/one-hot se hace
solo con datos de train en cada fold de cross-validation (evita data
leakage).

In [ ]:
train_sub, val_sub = data.train_val_split(train_df, test_size=0.2)
X_train, y_train = data.get_features_and_target(train_sub)
X_val, y_val = data.get_features_and_target(val_sub)

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
print("Shape de X_train luego del preprocesamiento:", X_train_transformed.shape)


## 4. Baseline

Antes de entrenar modelos "reales", establecemos un piso de comparación
con un clasificador trivial (predice siempre la clase más frecuente /
predicciones aleatorias respetando la distribución de clases).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_val)
baseline_f1 = f1_score(y_val, baseline_preds, average="macro")
print(f"Baseline (DummyClassifier) F1-Macro en validación: {baseline_f1:.4f}")


## 5. Modelos + búsqueda de hiperparámetros (F1-Macro)

Entrenamos al menos dos modelos de familias distintas
(`src/spotify_clf/models.py` trae logistic regression, random forest,
hist gradient boosting, KNN y, si `xgboost` está instalado, XGBoost).
Para cada uno corremos `RandomizedSearchCV`/`GridSearchCV` optimizando
**F1-Macro** (la métrica oficial de la competencia), con
`StratifiedKFold` para preservar la proporción de géneros en cada fold.

Nota: correr la búsqueda para todos los modelos del registro puede
demorar. Para una iteración rápida, arranquen con
`models_to_run = ["logistic_regression", "random_forest"]` y agreguen
más modelos/mayor `n_iter` una vez que el pipeline esté validado.

In [ ]:
registry = get_model_registry()
print("Modelos disponibles:", list(registry.keys()))

models_to_run = ["logistic_regression", "random_forest", "hist_gradient_boosting"]
search_results = {}

for model_name in models_to_run:
    spec = registry[model_name]
    print(f"\n=== Tuning {model_name} ===")
    search = tune_model(
        spec,
        X_train,
        y_train,
        search_type="random",
        n_iter=15,
        cv_folds=5,
    )
    search_results[model_name] = search
    print(f"Mejor F1-Macro (CV): {search.best_score_:.4f}")
    print(f"Mejores hiperparámetros: {search.best_params_}")


In [ ]:
from sklearn.metrics import classification_report

for model_name, search in search_results.items():
    preds = search.best_estimator_.predict(X_val)
    val_f1 = f1_score(y_val, preds, average="macro")
    print(f"{model_name}: F1-Macro en validación = {val_f1:.4f}")


## 6. Model Selection (cross-validation)

Comparamos los mejores estimadores de cada familia con cross-validation
sobre el set de train completo (no solo el split train/val de arriba),
para una estimación más robusta de F1-Macro antes de elegir el modelo
final.

In [ ]:
X_full, y_full = data.get_features_and_target(train_df)

eval_results = [
    cross_validate_model(name, search.best_estimator_, X_full, y_full, cv_folds=5)
    for name, search in search_results.items()
]

comparison_df = results_to_dataframe(eval_results)
comparison_df


El modelo con mayor `f1_macro_mean` (y desvío estándar razonable entre
folds) es el candidato a elegir para la competencia de Kaggle. Documenten
acá cuál eligieron y por qué (F1 medio, estabilidad entre folds, tiempo de
entrenamiento, interpretabilidad, etc.).

## 7. MAP@3: implementación y comparación con F1-Macro

`src/spotify_clf/metrics.py` implementa `mean_average_precision_at_k`
sobre las top-k predicciones ordenadas por probabilidad
(`top_k_predictions`, usa `predict_proba`). `cross_validate_model` ya la
calculó en la tabla anterior (`map_at_3_mean`) usando la misma partición
de CV que F1-Macro, así que son directamente comparables.

In [ ]:
comparison_df[["model", "f1_macro_mean", "map_at_3_mean"]]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison_df))
width = 0.35
ax.bar(x - width / 2, comparison_df["f1_macro_mean"], width, label="F1-Macro")
ax.bar(x + width / 2, comparison_df["map_at_3_mean"], width, label="MAP@3")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["model"], rotation=20)
ax.set_ylabel("score")
ax.set_title("F1-Macro vs MAP@3 por modelo")
ax.legend()
plt.tight_layout()
plt.show()


**Análisis esperado** _(completar)_: ¿el ranking de modelos cambia según
la métrica? Es razonable que MAP@3 sea sistemáticamente más alto que
F1-Macro (le alcanza con acertar en el top-3), pero lo interesante es si
**el modelo elegido por F1-Macro también es el mejor en MAP@3**, o si
conviene un modelo distinto según el objetivo de negocio.

## 8. Optimizando pensando en MAP@3

Algunas estrategias para mejorar MAP@3 aunque implique resignar algo de
F1-Macro (a evaluar empíricamente con el código de abajo):

1. **Calibrar las probabilidades** (`CalibratedClassifierCV`): mejora el
   ranking relativo de las clases en `predict_proba`, que es justamente lo
   que necesita MAP@3.
2. **`class_weight="balanced"`** (o resampling): un modelo que no fuerza
   tanto la frontera entre clases parecidas puede dejar el género correcto
   en 2do/3er lugar más seguido, aunque falle más en el top-1.
3. **Ensamble por promedio de probabilidades** entre modelos de familias
   distintas: un género que un modelo no llega a poner primero puede
   quedar mejor rankeado al promediar con otro modelo.
4. **Elegir el modelo/hiperparámetros por MAP@3 en vez de F1-Macro** en la
   búsqueda de hiperparámetros, usando `scoring=map_at_3_scorer` en
   `tune_model`.

In [ ]:
# Ejemplo: re-tunear el mejor modelo de la sección 5, pero optimizando
# MAP@3 en lugar de F1-Macro, para ver si cambian los hiperparámetros
# elegidos y si mejora MAP@3 en validación.
best_model_name = comparison_df.iloc[0]["model"]
spec = registry[best_model_name]

search_map3 = tune_model(
    spec,
    X_train,
    y_train,
    search_type="random",
    n_iter=15,
    cv_folds=5,
    scoring=map_at_3_scorer,
)

print(f"Mejor MAP@3 (CV) optimizando directamente MAP@3: {search_map3.best_score_:.4f}")
print(f"Hiperparámetros: {search_map3.best_params_}")

# Comparación en el set de validación local
model_f1_opt = search_results[best_model_name].best_estimator_
model_map3_opt = search_map3.best_estimator_

for label, model in [("optimizado para F1-Macro", model_f1_opt), ("optimizado para MAP@3", model_map3_opt)]:
    val_preds = model.predict(X_val)
    val_f1 = f1_score(y_val, val_preds, average="macro")
    val_map3 = map_at_3_from_model(model, X_val, y_val)
    print(f"{label}: F1-Macro={val_f1:.4f} | MAP@3={val_map3:.4f}")


**Conclusión** _(completar)_: ¿vale la pena, para este dataset, resignar
F1-Macro a cambio de MAP@3? ¿Cuánto se gana/pierde en cada métrica?

## 9. Discusión: ¿qué métrica usar según el caso de negocio?

_(Desarrollar con sus propias conclusiones — puntos de partida abajo)_

- **F1-Macro (top-1 forzado)** tiene sentido cuando el sistema debe tomar
  una única decisión automática sin intervención humana y todos los
  errores son igual de costosos (ej. etiquetar metadata interna de
  catálogo, moderación de contenido, clasificación para reporting).
- **MAP@3 (ranking, top-3)** es más fiel al caso de uso real de un
  sistema de recomendación: al usuario se le muestra una lista de
  opciones (ej. "más canciones de este estilo"), y que el género correcto
  esté en el top-3 ya genera una buena experiencia, aunque no sea la
  primera opción mostrada.
- Otras métricas a considerar según el caso real de negocio:
  - **Precision@k / Recall@k** si interesa medir específicamente cuántas
    de las top-k recomendaciones son relevantes (no solo si la correcta
    apareció).
  - **NDCG (Normalized Discounted Cumulative Gain)** si, además de si el
    género correcto está en el top-k, importa en qué posición exacta
    (pondera más las posiciones más altas que MAP@k con distinto peso).
  - **Métricas por clase / F1 ponderado (weighted)** si algunos géneros
    son mucho más importantes para el negocio que otros (p. ej. géneros
    con mayor volumen de escucha o mayor monetización), en vez de tratar
    a todos los géneros por igual como hace F1-Macro.
  - **Métricas de negocio directas** (engagement, tiempo de escucha,
    skip-rate de las recomendaciones) si se pudiera hacer un A/B test:
    en definitiva, F1-Macro y MAP@3 son proxies de laboratorio, y la
    consigna pide justamente contrastarlas con el impacto real.

## 10. Entrenamiento final y submission para Kaggle

Reentrenamos el modelo elegido sobre **todo** el `train.csv` (sin el
split de validación) y generamos el CSV a subir a Kaggle. Ajusten
`final_model` según la conclusión de las secciones anteriores (el mejor
en F1-Macro, o el optimizado para MAP@3 si decidieron priorizarlo).

In [ ]:
final_model_name = best_model_name  # cambiar si se elige otro modelo
final_pipeline = registry[final_model_name].pipeline.set_params(
    **search_results[final_model_name].best_params_
)
final_pipeline.fit(X_full, y_full)

import joblib
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, config.MODELS_DIR / f"{final_model_name}_final.joblib")
print(f"Modelo final ({final_model_name}) entrenado sobre todo el train y guardado.")


In [ ]:
X_test = test_df.drop(
    columns=[c for c in [config.ID_COL, *config.TEXT_COLS] if c in test_df.columns]
)
test_predictions = final_pipeline.predict(X_test)

submission_df = build_submission(test_df[config.ID_COL], test_predictions)
out_path = save_submission(submission_df)
print(f"Submission guardada en: {out_path}")
submission_df.head()


## 11. Conclusiones finales

_(completar)_: resumen de decisiones de preprocesamiento, modelo elegido,
resultado en la competencia de Kaggle (F1-Macro público/privado), y la
recomendación final sobre F1-Macro vs. MAP@3 para el caso de negocio de
Spotify.